In [4]:
import pandas as pd

## 1. Datos de entrada
# --- BLOQUE 1: DATAFRAME DE 7 EMPRESAS---
datos = {
    'nombre': ['Ferretería García','Bar El Rincón','Peluquería Ana','Tienda Moda Sol','Gestoría Pérez', 'Clínica Dental Martínez','Academia Idiomas Sol', 'Clínica Dental García','Academia de Idiomas English House'],
    'sector': ['Ferretería','Hostelería','Peluquería','Moda','Gestoría','Salud','Educación','Salud','Educación'],
    'facturacion': [320000, 180000, 95000, 210000, 140000, 280000,160000,295000,155000],
    'coste_ventas': [210000, 95000, 38000, 140000, 45000,84000,48000,118000,77500],
    'empleados': [5, 4, 2, 3, 3,6,5,6,5],
    'deuda': [45000, 12000, 8000, 35000, 5000,90000,15000,95000,14000],
    'activo_total': [180000, 85000, 40000, 120000, 70000,220000,75000,230000,72000],
    'año_fundacion': [2008, 2015, 2019, 2012, 2006,2014,2016,2013,2017]
}

df = pd.DataFrame(datos)
print(df)
print('\nNúmero de empresas analizadas:', len(df))

## 2. Cálculo de ratios
# --- BLOQUE 2: CÁLCULO DE RATIOS FINANCIEROS ---
# pandas calcula el ratio para TODAS las empresas a la vez con una sola línea
# es lo mismo que hacías con el diccionario pero para las 5 empresas simultáneamente

# Margen bruto en porcentaje
df['margen_bruto_pct'] = ((df['facturacion'] - df['coste_ventas']) / df['facturacion'] * 100).round(1)

# Ratio de endeudamiento (deuda / lo que tiene la empresa)
df['ratio_endeudamiento'] = (df['deuda'] / df['activo_total']).round(2)

# Facturación por empleado — eficiencia del negocio
df['fact_por_empleado'] = (df['facturacion'] / df['empleados']).round(0)

# Años que lleva abierta la empresa
df['años_activo'] = 2026 - df['año_fundacion']

# Ver la tabla con los nuevos ratios calculados
print(df[['nombre', 'margen_bruto_pct', 'ratio_endeudamiento', 'fact_por_empleado', 'años_activo']])

# --- BLOQUE 3: FILTROS DE INVERSIÓN ---
# Usamos los ratios calculados arriba para filtrar empresas
# df[condición] devuelve solo las filas donde la condición es True

# Empresas con bajo riesgo de deuda (menos del 30%)
bajo_riesgo = df[df['ratio_endeudamiento'] < 0.3]
print('Empresas con bajo endeudamiento:')
print(bajo_riesgo[['nombre', 'ratio_endeudamiento']])

# Empresas con buen margen Y bajo riesgo a la vez
# & significa que las dos condiciones tienen que cumplirse
candidatas = df[(df['margen_bruto_pct'] > 25) & (df['ratio_endeudamiento'] < 0.4)]
print('\nMejores candidatas para inversión:')
print(candidatas[['nombre', 'margen_bruto_pct', 'ratio_endeudamiento']])

# Ordenar todas las empresas por margen de mayor a menor
# ascending=False significa de mayor a menor
ranking = df.sort_values('margen_bruto_pct', ascending=False)
print('\nRanking por margen bruto:')
print(ranking[['nombre', 'margen_bruto_pct']])

# Empresas que están por encima de la media del grupo
media_margen = df['margen_bruto_pct'].mean()
sobre_media = df[df['margen_bruto_pct'] > media_margen]
print('\nEmpresas sobre la media del grupo:')
print(sobre_media['nombre'].tolist())

# --- BLOQUE 4: SCORING AUTOMÁTICO CON pd.cut() ---
# Versión mejorada del scoring de la Semana 1 — calcula para todas las empresas a la vez

# Scoring de endeudamiento
bins_deuda = [0, 0.25, 0.5, float('inf')]
labels_deuda = ['Bajo', 'Medio', 'Alto']
df['riesgo_deuda'] = pd.cut(df['ratio_endeudamiento'], bins=bins_deuda, labels=labels_deuda)

# Scoring de margen
bins_margen = [float('-inf'), 15, 30, float('inf')]
labels_margen = ['Bajo', 'Medio', 'Alto']
df['calidad_margen'] = pd.cut(df['margen_bruto_pct'], bins=bins_margen, labels=labels_margen)

# Puntuación numérica combinada
# Margen alto = 3 puntos, medio = 2, bajo = 1
# Deuda baja = 3 puntos, media = 2, alta = 1
# Más de 7 años activa = 2 puntos, menos = 1
mapa_margen = {'Bajo': 1, 'Medio': 2, 'Alto': 3}
mapa_deuda  = {'Bajo': 3, 'Medio': 2, 'Alto': 1}

df['puntos_margen']     = df['calidad_margen'].map(mapa_margen).astype(int)
df['puntos_deuda']      = df['riesgo_deuda'].map(mapa_deuda).astype(int)
df['puntos_antiguedad'] = df['años_activo'].apply(lambda x: 2 if x >= 7 else 1)
df['puntuacion_total']  = df['puntos_margen'] + df['puntos_deuda'] + df['puntos_antiguedad']

print(df[['nombre', 'calidad_margen', 'riesgo_deuda', 'años_activo', 'puntuacion_total']])

# --- BLOQUE 5: RANKING DE OPORTUNIDADES ---
# Ordenar todas las empresas por puntuación total de mayor a menor
# reset_index(drop=True) reinicia el índice desde 0 tras ordenar
ranking = df.sort_values('puntuacion_total', ascending=False).reset_index(drop=True)

# Empezar el índice en 1 en lugar de 0 — más legible como ranking
ranking.index = ranking.index + 1

# Columnas que aparecen en el ranking final
cols_ranking = ['nombre', 'sector', 'facturacion', 'margen_bruto_pct',
                'ratio_endeudamiento', 'años_activo', 'puntuacion_total',
                'calidad_margen', 'riesgo_deuda']

print('=' * 55)
print('RANKING DE OPORTUNIDADES DE INVERSIÓN EN PYMEs')
print('=' * 55)
print(ranking[cols_ranking].to_string())
print('\nMejor oportunidad:', ranking.iloc[0]['nombre'])
print('Mayor riesgo:     ', ranking.iloc[-1]['nombre'])


## 3. Estadística Descriptiva
# --- BLOQUE 6: RESUMEN ESTADÍSTICO Y AGRUPACIÓN ---

# describe() da un resumen estadístico completo de todas las columnas numéricas
# muestra mínimo, máximo, media, mediana y desviación estándar de golpe
print('=== RESUMEN ESTADÍSTICO ===')
print(df[['margen_bruto_pct', 'ratio_endeudamiento', 'fact_por_empleado']].describe())

# Media de cada ratio para todo el conjunto
# sirve para saber si una empresa está por encima o por debajo de la media del grupo
print('\n=== MEDIAS DEL GRUPO ===')
print('Margen bruto medio:', df['margen_bruto_pct'].mean().round(1), '%')
print('Endeudamiento medio:', df['ratio_endeudamiento'].mean().round(2))
print('Facturación por empleado media:', df['fact_por_empleado'].mean().round(0), '€')

# Qué empresa tiene el mejor y peor margen
# idxmax() devuelve el índice de la fila con el valor más alto
print('\n=== DESTACADOS ===')
print('Mejor margen:', df.loc[df['margen_bruto_pct'].idxmax(), 'nombre'])
print('Peor margen:', df.loc[df['margen_bruto_pct'].idxmin(), 'nombre'])
print('Menor riesgo de deuda:', df.loc[df['ratio_endeudamiento'].idxmin(), 'nombre'])

# groupby agrupa las filas por sector y calcula la media de cada grupo
# útil para comparar sectores entre sí — la base del análisis sectorial
print('\n=== MEDIAS POR SECTOR ===')
por_sector = df.groupby('sector')[['margen_bruto_pct', 'ratio_endeudamiento']].mean().round(2)
print(por_sector)
print('\nSectores ordenados por margen (mejor a peor):')
print(por_sector.sort_values('margen_bruto_pct', ascending=False))


# --- BLOQUE 7: RESUMEN DEL ANÁLISIS ---
# Resumen ejecutivo del notebook — lo que le mostrarías a un inversor
# en la primera página de un informe antes de que lea la tabla completa

print('=== ANALIZADOR DE PYMEs v0.1 — RESUMEN ===')

# Total de empresas en el DataFrame
print(f'Empresas analizadas: {len(df)}')

# Número de sectores distintos — nunique() cuenta valores únicos
print(f'Sectores cubiertos: {df["sector"].nunique()}')

# Mejor empresa — primera fila del ranking (mayor puntuación)
print(f'Mejor oportunidad: {ranking.iloc[0]["nombre"]} ({ranking.iloc[0]["puntuacion_total"]} puntos)')

# Peor empresa — última fila del ranking (menor puntuación)
print(f'Mayor riesgo: {ranking.iloc[-1]["nombre"]} ({ranking.iloc[-1]["puntuacion_total"]} puntos)')

# Margen bruto medio del conjunto
print(f'Margen bruto medio del grupo: {df["margen_bruto_pct"].mean().round(1)}%')

# Sector con mejor margen medio
print(f'Sector más rentable: {df.groupby("sector")["margen_bruto_pct"].mean().idxmax()}')


# --- BLOQUE 8: EXPORTACIÓN A EXCEL ---
!pip install openpyxl -q

import openpyxl

nombre_archivo = 'ranking_pymes_v01.xlsx'

with pd.ExcelWriter(nombre_archivo, engine='openpyxl') as writer:

    # Hoja 1: Ranking completo ordenado por puntuación
    ranking[cols_ranking].to_excel(writer, sheet_name='Ranking', index=True)

    # Hoja 2: Datos originales con todos los ratios calculados
    df.to_excel(writer, sheet_name='Datos', index=False)

print(f'Archivo guardado: {nombre_archivo}')
print('Hojas: Ranking, Datos')

# --- BLOQUE 9: ESTADÍSTICA DESCRIPTIVA ---

print('=== RESUMEN ESTADÍSTICO DEL CONJUNTO ===')
print(df[['margen_bruto_pct','ratio_endeudamiento','fact_por_empleado','años_activo']].describe().round(2))

print('\n=== VALORES CLAVE ===')
print('Margen bruto medio:', df['margen_bruto_pct'].mean().round(1), '%')
print('Mediana de margen: ', round(df['margen_bruto_pct'].median(), 1), '%')
print('Desviación estándar:', round(df['margen_bruto_pct'].std(), 1), '%')

# Detectar outliers — empresas más de 1.5 desviaciones de la media
media   = df['margen_bruto_pct'].mean()
std_dev = df['margen_bruto_pct'].std()
outliers = df[abs(df['margen_bruto_pct'] - media) > 1.5 * std_dev]
print('\nOutliers por margen bruto:')
print(outliers[['nombre', 'margen_bruto_pct']])

## 4. Análisis de correlaciones
# --- BLOQUE 10: MATRIZ DE CORRELACIONES ---

columnas_num = ['facturacion', 'margen_bruto_pct', 'ratio_endeudamiento',
                'fact_por_empleado', 'años_activo', 'puntuacion_total']

correlacion = df[columnas_num].corr().round(2)
print('=== MATRIZ DE CORRELACIÓN ===')
print(correlacion)

print('\nCorrelación de cada variable con la puntuación total:')
print(correlacion['puntuacion_total'].sort_values(ascending=False))

for col in columnas_num:
    val = correlacion.loc[col, 'puntuacion_total']
    if col != 'puntuacion_total' and abs(val) > 0.5:
        direccion = 'positiva' if val > 0 else 'negativa'
        print(f'{col} tiene correlación {direccion} fuerte con puntuación: {val}')

## 5. Análisis estadístico por sector
# --- BLOQUE 11: ANÁLISIS ESTADÍSTICO POR SECTOR ---

stats_sector = df.groupby('sector')['margen_bruto_pct'].agg(
    media='mean',
    mediana='median',
    desviacion='std',
    minimo='min',
    maximo='max',
    empresas='count'
).round(2)

print('=== ANÁLISIS ESTADÍSTICO POR SECTOR ===')
print(stats_sector)

print('\nSectores ordenados por margen medio:')
print(stats_sector.sort_values('media', ascending=False))

print('\nSectores por variabilidad (std):')
print(stats_sector[['desviacion']].sort_values('desviacion', ascending=False))

## 6. Scoring estadístico por percentiles
# --- BLOQUE 12: SCORING POR PERCENTILES ---

# Para margen: más es mejor
df['pct_margen'] = df['margen_bruto_pct'].rank(pct=True).round(2)

# Para endeudamiento: menos es mejor — rank invertido
df['pct_deuda'] = (1 - df['ratio_endeudamiento'].rank(pct=True)).round(2)

# Para antigüedad: más es mejor
df['pct_antiguedad'] = df['años_activo'].rank(pct=True).round(2)

# Pesos: margen 50%, deuda 30%, antigüedad 20%
df['score_estadistico'] = (
    df['pct_margen']     * 0.50 +
    df['pct_deuda']      * 0.30 +
    df['pct_antiguedad'] * 0.20
).round(3)

ranking_v2 = df.sort_values('score_estadistico', ascending=False).reset_index(drop=True)
ranking_v2.index = ranking_v2.index + 1

print('=== RANKING v0.2 — SCORING ESTADÍSTICO ===')
print(ranking_v2[['nombre', 'pct_margen', 'pct_deuda', 'pct_antiguedad', 'score_estadistico']])
print('\nMejor oportunidad:', ranking_v2.iloc[0]['nombre'])

## 7. Ranking final v0.2
# --- BLOQUE 13: RANKING v0.2 ---

print('=' * 55)
print('RANKING v0.2 — SCORING ESTADÍSTICO POR PERCENTILES')
print('=' * 55)

cols_ranking_v2 = ['nombre', 'sector', 'pct_margen', 'pct_deuda',
                   'pct_antiguedad', 'score_estadistico']

print(ranking_v2[cols_ranking_v2].to_string())
print('\nMejor oportunidad:', ranking_v2.iloc[0]['nombre'],
      '— score:', ranking_v2.iloc[0]['score_estadistico'])
print('Mayor riesgo:     ', ranking_v2.iloc[-1]['nombre'],
      '— score:', ranking_v2.iloc[-1]['score_estadistico'])

# --- BLOQUE 14: RESUMEN FINAL v0.2 ---

print('==== ANALIZADOR DE PYMEs v0.2 — RESUMEN ====')
print(f'Empresas analizadas: {len(df)}')
print(f'Sectores cubiertos: {df["sector"].nunique()}')
print(f'Correlación margen-score: {round(df["margen_bruto_pct"].corr(df["score_estadistico"]), 2)}')
print(f'Mejor oportunidad: {ranking_v2.iloc[0]["nombre"]} (score: {ranking_v2.iloc[0]["score_estadistico"]})')
print(f'Mayor riesgo: {ranking_v2.iloc[-1]["nombre"]} (score: {ranking_v2.iloc[-1]["score_estadistico"]})')
print(f'Sector con mejor margen medio: {stats_sector["media"].idxmax()}')

                              nombre      sector  facturacion  coste_ventas  \
0                  Ferretería García  Ferretería       320000        210000   
1                      Bar El Rincón  Hostelería       180000         95000   
2                     Peluquería Ana  Peluquería        95000         38000   
3                    Tienda Moda Sol        Moda       210000        140000   
4                     Gestoría Pérez    Gestoría       140000         45000   
5            Clínica Dental Martínez       Salud       280000         84000   
6               Academia Idiomas Sol   Educación       160000         48000   
7              Clínica Dental García       Salud       295000        118000   
8  Academia de Idiomas English House   Educación       155000         77500   

   empleados  deuda  activo_total  año_fundacion  
0          5  45000        180000           2008  
1          4  12000         85000           2015  
2          2   8000         40000           2019  
3    